<a href="https://colab.research.google.com/github/PathakDeepak/TensorFlow_Practise/blob/main/Week_33_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<center><font color="brown" size="6"><b>Week 33: Graded Mini Project</b></font></center>

**Project title**: Grounded Answers from a Small Knowledge Pack

**Objective**: Apply your understanding of Retrieval-Augmented Generation (RAG), prompt
engineering, and conversation memory to design and evaluate a lightweight questionanswering pipeline over a small, local knowledge base. Experiment with chunking strategies,
embedding models, and prompt templates to evaluate how these design choices affect
grounding, citation accuracy, and conversational coherence.

### Install Dependencies

In [50]:
!pip install -q sentence-transformers faiss-cpu openai pandas

## <font color="brown">Task 1: Data prep (tiny corpus)</font>

In [59]:
import os

# Create dataset folder
os.makedirs("/content/docs", exist_ok=True)

# Sample documents
documents_data = {
    "doc1_rag.txt": "RAG combines retrieval with generation using external knowledge.",
    "doc2_embeddings.txt": "Embeddings convert text into vectors for similarity search.",
    "doc3_faiss.txt": "FAISS is used for efficient similarity search in vector space.",
    "doc4_llm.txt": "Large Language Models generate natural language responses.",
    "doc5_memory.txt": "Conversation memory helps maintain context across interactions."
}


for filename, content in documents_data.items():
    with open(f"/content/docs/{filename}", "w") as f:
        f.write(content)


documents = []

for file in os.listdir("/content/docs"):
    with open(f"/content/docs/{file}", "r") as f:
        text = f.read()

    documents.append({
        "text": text,
        "metadata": {
            "doc_title": file,
            "page": 1,
            "chunk_id": 0
        }
    })

print(documents)

[{'text': 'Embeddings convert text into vectors for similarity search.', 'metadata': {'doc_title': 'doc2_embeddings.txt', 'page': 1, 'chunk_id': 0}}, {'text': 'RAG combines retrieval with generation using external knowledge.', 'metadata': {'doc_title': 'doc1_rag.txt', 'page': 1, 'chunk_id': 0}}, {'text': 'FAISS is used for efficient similarity search in vector space.', 'metadata': {'doc_title': 'doc3_faiss.txt', 'page': 1, 'chunk_id': 0}}, {'text': 'Large Language Models generate natural language responses.', 'metadata': {'doc_title': 'doc4_llm.txt', 'page': 1, 'chunk_id': 0}}, {'text': 'Conversation memory helps maintain context across interactions.', 'metadata': {'doc_title': 'doc5_memory.txt', 'page': 1, 'chunk_id': 0}}]


## <font color="brown">Task 2: Baseline pipeline</font>

In [52]:
def build_chunks(chunk_size, overlap):
    all_chunks = []

    for doc in documents:
        chunks = chunk_text(doc["text"], chunk_size, overlap)
        for i, chunk in enumerate(chunks):
            all_chunks.append({
                "text": chunk,
                "metadata": {
                    "doc_title": doc["metadata"]["doc_title"],
                    "page": doc["metadata"]["page"],
                    "chunk_id": i
                }
            })
    return all_chunks


def build_index(chunks):
    texts = [c["text"] for c in chunks]
    embeddings = model.encode(texts)

    dim = embeddings.shape[1]
    idx = faiss.IndexFlatL2(dim)
    idx.add(np.array(embeddings))

    return idx, texts


def retrieve(query, index, chunks, k=3):
    q_emb = model.encode([query])
    D, I = index.search(q_emb, k)
    return [chunks[i] for i in I[0]]


def rag_pipeline(query, prompt_template, index, chunks):
    retrieved = retrieve(query, index, chunks)

    context = "\n".join([c["text"] for c in retrieved])

    citations = [
        f"[{c['metadata']['doc_title']}#{c['metadata']['page']}]"
        for c in retrieved
    ]

    history = "\n".join([f"Q:{q} A:{a}" for q, a in conversation_memory[-2:]])

    prompt = prompt_template(context + "\n" + history, query)

    answer = generate(prompt)

    conversation_memory.append((query, answer))

    return {
        "answer": answer,
        "citations": citations,
        "chunks": retrieved
    }

## <font color="brown">Task 3: Prompt templates (2 variants)</font>

In [53]:
def prompt_p1(context, question):
    return f"""
Answer in ≤120 tokens.
Use only provided context.
Add citations like [title#page].
If unsure, say "I don't know".

Context:
{context}

Question:
{question}
"""


def prompt_p2(context, question):
    return f"""
Use only provided context.

Evidence:
- Extract 2 supporting facts

Answer:
- Final concise answer (≤120 tokens)
- Include citations [title#page]

Context:
{context}

Question:
{question}
"""

## <font color="brown">Task 4: Experiments grid</font>

In [20]:
experiment_configs = [
    {"run_id": "run1", "chunk_size": 300, "overlap": 0.1, "prompt": prompt_p1},
    {"run_id": "run2", "chunk_size": 600, "overlap": 0.15, "prompt": prompt_p1},
    {"run_id": "run3", "chunk_size": 1000, "overlap": 0.2, "prompt": prompt_p2},
    {"run_id": "run4", "chunk_size": 300, "overlap": 0.2, "prompt": prompt_p2},
    {"run_id": "run5", "chunk_size": 600, "overlap": 0.1, "prompt": prompt_p2},
    {"run_id": "run6", "chunk_size": 1000, "overlap": 0.1, "prompt": prompt_p1},
]

## <font color="brown">Task 5: Question set (manual event)</font>

In [26]:
questions = [
    # factual
    "What is RAG?",
    "What are embeddings?",
    "What is FAISS used for?",
    "What is an LLM?",
    "What is conversation memory?",
    "How does retrieval work?",

    # boundary
    "What is quantum computing?",
    "Who is the president of USA?",

    # follow-up
    "Explain RAG again",
    "How does memory help in RAG?"
]

In [35]:
def run_experiment(config):
    run_id = config["run_id"]

    for q in questions:
        start = time.time()

        result = rag_pipeline(q, config["prompt"])

        latency = time.time() - start

        scores = evaluate(result["answer"])

        record = {
            "run_id": run_id,
            "question": q,
            "answer": result["answer"],
            "citations": result["citations"],
            "latency": latency,
            **scores
        }

        log_result(record)

In [36]:
for config in experiment_configs:
    run_experiment(config)

## <font color="brown">Task 6: Logging</font>

In [54]:
def log_result(record):
    with open("/content/logs.jsonl", "a") as f:
        f.write(json.dumps(record) + "\n")

## <font color="brown">Task 7: Manual evaluation (simple rubric below)</font>

In [55]:
def evaluate(answer, question, chunks):
    ans = answer.lower()

    # relevance
    relevance = 1 if any(word in ans for word in question.lower().split()) else 0

    # groundedness
    chunk_text = " ".join([c["text"].lower() for c in chunks])
    groundedness = 1 if any(word in chunk_text for word in ans.split()) else 0

    # citation
    citation_correctness = 1 if "[" in answer else 0

    # conciseness
    conciseness = 1 if len(answer.split()) <= 120 else 0

    # memory
    memory_continuity = 1 if "again" in question.lower() else 0

    return {
        "relevance": relevance,
        "groundedness": groundedness,
        "citation_correctness": citation_correctness,
        "conciseness": conciseness,
        "memory_continuity": memory_continuity
    }

### Run Experiments

In [56]:
def run_experiment(config):
    global conversation_memory
    conversation_memory = []

    run_id = config["run_id"]

    chunks = build_chunks(config["chunk_size"], config["overlap"])
    index, _ = build_index(chunks)

    for q in questions:
        start = time.time()

        result = rag_pipeline(q, config["prompt"], index, chunks)

        latency = time.time() - start

        scores = evaluate(result["answer"], q, result["chunks"])

        record = {
            "run_id": run_id,
            "question": q,
            "answer": result["answer"],
            "citations": result["citations"],
            "used_chunks": [c["metadata"] for c in result["chunks"]],
            "latency": latency,
            "tokens": len(result["answer"].split()),
            **scores
        }

        log_result(record)

In [57]:
open("/content/logs.jsonl", "w").close()

for config in experiment_configs:
    run_experiment(config)

## <font color="brown">Task 8: Summary report (2-3 pages)</font>

In [58]:
import pandas as pd

df = pd.read_json("/content/logs.jsonl", lines=True)

# compute score
score_cols = [
    "relevance",
    "groundedness",
    "citation_correctness",
    "conciseness",
    "memory_continuity"
]

df["overall_score"] = df[score_cols].mean(axis=1)

summary = df.groupby("run_id")[score_cols + ["latency"]].mean()

print("\n=== TABLE OF RUNS ===")
print(summary)


summary["overall_score"] = summary.mean(axis=1)
best_run = summary["overall_score"].idxmax()

print("\nBest run:", best_run)

wins = (
    df.sort_values(by="overall_score", ascending=False)
      .drop_duplicates("question")
      .head(2)
)


failures = (
    df.sort_values(by="overall_score")
      .drop_duplicates("question")
      .head(2)
)

print("\n=== WINS ===")
for _, row in wins.iterrows():
    print("\nQ:", row["question"])
    print("A:", row["answer"])

print("\n=== FAILURES ===")
for _, row in failures.iterrows():
    print("\nQ:", row["question"])
    print("A:", row["answer"])


=== TABLE OF RUNS ===
        relevance  groundedness  citation_correctness  conciseness  \
run_id                                                               
run1          0.7           1.0                   0.7          1.0   
run2          0.7           1.0                   0.8          1.0   
run3          1.0           1.0                   1.0          1.0   
run4          1.0           1.0                   1.0          1.0   
run5          1.0           1.0                   1.0          1.0   
run6          0.7           1.0                   0.8          1.0   

        memory_continuity   latency  
run_id                               
run1                  0.1  1.542509  
run2                  0.1  1.519751  
run3                  0.1  2.831233  
run4                  0.1  2.557202  
run5                  0.1  2.733346  
run6                  0.1  1.624573  

Best run: run3

=== WINS ===

Q: Explain RAG again
A: RAG combines retrieval with generation by utilizing exter